# Group16 Feedback Prize ELL Inference Notebook

Submission entry for the CA6125 LLM and RAG course project. The notebook reproduces the stacked ensemble used in the local report on the official Kaggle test set provided in `/kaggle/input`.

Pipeline:
1. Word and character TF-IDF features fitted on train + test text.
2. Three component models trained with the same fold split.
3. Per-target convex blend selected on out-of-fold predictions.
4. Final scores clipped to [1.0, 5.0] and written to `submission.csv`.

All code is original work for Group16. Public Kaggle solutions are referenced in the project report but no source files are copied.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMRegressor

SEED = 42
N_SPLITS = 5
INPUT_DIR = Path('/kaggle/input/feedback-prize-english-language-learning')
TARGETS = ['cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions']

train = pd.read_csv(INPUT_DIR / 'train.csv')
test = pd.read_csv(INPUT_DIR / 'test.csv')
submission = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
print(train.shape, test.shape, submission.shape)

In [ ]:
def add_text_stats(df):
    text = df['full_text'].fillna('').astype(str)
    out = pd.DataFrame()
    out['char_count'] = text.str.len()
    out['word_count'] = text.str.split().map(len)
    out['sentence_count'] = text.str.count(r'[.!?]+').clip(lower=1)
    out['paragraph_count'] = text.str.count(r'\n+').add(1)
    out['avg_word_len'] = text.map(lambda v: float(np.mean([len(w) for w in v.split()])) if v.split() else 0.0)
    out['comma_count'] = text.str.count(',')
    out['semicolon_count'] = text.str.count(';')
    out['quote_count'] = text.str.count('"')
    out['uppercase_ratio'] = text.map(lambda v: sum(c.isupper() for c in v) / max(1, sum(c.isalpha() for c in v)))
    out['digit_ratio'] = text.map(lambda v: sum(c.isdigit() for c in v) / max(1, len(v)))
    return out.astype(float).to_numpy()

vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2,
                              max_features=60000, sublinear_tf=True, strip_accents='unicode')),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                              max_features=80000, sublinear_tf=True)),
])
all_text = pd.concat([train['full_text'], test['full_text']], axis=0).fillna('').astype(str)
tfidf_all = vectorizer.fit_transform(all_text)
tfidf_train = tfidf_all[:len(train)]
tfidf_test = tfidf_all[len(train):]

stats_train = add_text_stats(train)
stats_test = add_text_stats(test)
scaler = StandardScaler().fit(stats_train)
stats_train_s = scaler.transform(stats_train)
stats_test_s = scaler.transform(stats_test)

fused_train = sparse.hstack([tfidf_train, sparse.csr_matrix(stats_train_s)]).tocsr()
fused_test = sparse.hstack([tfidf_test, sparse.csr_matrix(stats_test_s)]).tocsr()

svd = TruncatedSVD(n_components=128, random_state=SEED)
svd_train = svd.fit_transform(tfidf_train)
svd_test = svd.transform(tfidf_test)
lgbm_train = np.hstack([svd_train, stats_train_s])
lgbm_test = np.hstack([svd_test, stats_test_s])

y = train[TARGETS].to_numpy(dtype=float)
fold_signal = train[TARGETS].mean(axis=1).round(1).astype(str)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_assign = np.full(len(train), -1)
for fold, (_, val_idx) in enumerate(skf.split(train, fold_signal)):
    fold_assign[val_idx] = fold

In [ ]:
ALPHAS = [0.5, 1.0, 2.0, 4.0, 6.0, 10.0, 16.0, 24.0]

def ridge_per_target(x_train, y_train, x_valid, x_test, alphas, seed):
    rng = np.random.default_rng(seed)
    n = x_train.shape[0]
    inner = rng.permutation(n)
    cut = max(1, int(0.1 * n))
    in_val = inner[:cut]
    in_trn = inner[cut:]
    val_pred = np.zeros((x_valid.shape[0], len(TARGETS)))
    test_pred = np.zeros((x_test.shape[0], len(TARGETS)))
    for k in range(len(TARGETS)):
        best_alpha, best_score = alphas[0], np.inf
        for a in alphas:
            m = Ridge(alpha=float(a), random_state=seed).fit(x_train[in_trn], y_train[in_trn, k])
            s = float(np.sqrt(np.mean((y_train[in_val, k] - m.predict(x_train[in_val])) ** 2)))
            if s < best_score:
                best_alpha, best_score = float(a), s
        m = Ridge(alpha=best_alpha, random_state=seed).fit(x_train, y_train[:, k])
        val_pred[:, k] = m.predict(x_valid)
        test_pred[:, k] = m.predict(x_test)
    return val_pred, test_pred

def fit_component(x_train_full, x_test_full, alphas, name, seed):
    oof = np.zeros_like(y)
    test = np.zeros((x_test_full.shape[0], len(TARGETS)))
    for fold in range(N_SPLITS):
        trn_idx = np.where(fold_assign != fold)[0]
        val_idx = np.where(fold_assign == fold)[0]
        val_pred, fold_test = ridge_per_target(
            x_train_full[trn_idx], y[trn_idx], x_train_full[val_idx], x_test_full, alphas, seed + fold,
        )
        oof[val_idx] = val_pred
        test += fold_test / N_SPLITS
    rmse = np.sqrt(np.mean((y - oof) ** 2, axis=0))
    print(f'[{name}] CV MCRMSE = {rmse.mean():.5f}', dict(zip(TARGETS, rmse.round(4))))
    return oof, test

ridge_oof, ridge_test = fit_component(tfidf_train, tfidf_test, ALPHAS, 'ridge_tfidf_per_target', SEED)
fused_oof, fused_test = fit_component(fused_train, fused_test, [1.0, 2.0, 4.0, 6.0, 10.0, 16.0], 'ridge_fused', SEED + 1000)

In [ ]:
lgbm_oof = np.zeros_like(y)
lgbm_test = np.zeros((lgbm_test.shape[0], len(TARGETS)))
for fold in range(N_SPLITS):
    trn_idx = np.where(fold_assign != fold)[0]
    val_idx = np.where(fold_assign == fold)[0]
    fold_pred = np.zeros((len(val_idx), len(TARGETS)))
    fold_test = np.zeros((lgbm_test.shape[0], len(TARGETS)))
    for k in range(len(TARGETS)):
        m = LGBMRegressor(
            n_estimators=600, learning_rate=0.04, num_leaves=31,
            min_child_samples=20, subsample=0.85, subsample_freq=1,
            colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=0.1,
            random_state=SEED + k, verbosity=-1,
        )
        m.fit(lgbm_train[trn_idx], y[trn_idx, k])
        fold_pred[:, k] = m.predict(lgbm_train[val_idx])
        fold_test[:, k] = m.predict(lgbm_test)
    lgbm_oof[val_idx] = fold_pred
    lgbm_test += fold_test / N_SPLITS
rmse = np.sqrt(np.mean((y - lgbm_oof) ** 2, axis=0))
print('[lgbm_svd_fused] CV MCRMSE =', rmse.mean().round(5), dict(zip(TARGETS, rmse.round(4))))

In [ ]:
components_oof = [ridge_oof, fused_oof, lgbm_oof]
components_test = [ridge_test, fused_test, lgbm_test]
grid = np.linspace(0, 1, 21)
weight_combos = [(w0, w1, max(0.0, 1 - w0 - w1)) for w0 in grid for w1 in grid if w0 + w1 <= 1 + 1e-9]
stack_oof = np.zeros_like(y)
stack_test = np.zeros_like(ridge_test)
for k in range(len(TARGETS)):
    best_score, best_w = np.inf, weight_combos[0]
    for w in weight_combos:
        blend = sum(w[i] * components_oof[i][:, k] for i in range(3))
        s = float(np.sqrt(np.mean((y[:, k] - blend) ** 2)))
        if s < best_score:
            best_score, best_w = s, w
    stack_oof[:, k] = sum(best_w[i] * components_oof[i][:, k] for i in range(3))
    stack_test[:, k] = sum(best_w[i] * components_test[i][:, k] for i in range(3))
    print(TARGETS[k], 'weights', tuple(round(x, 2) for x in best_w), 'rmse', round(best_score, 4))
rmse = np.sqrt(np.mean((y - stack_oof) ** 2, axis=0))
print('[stacked_ensemble] CV MCRMSE =', rmse.mean().round(5), dict(zip(TARGETS, rmse.round(4))))

submission[TARGETS] = np.clip(stack_test, 1.0, 5.0)
submission.to_csv('submission.csv', index=False)
submission